In [0]:
import sys, os

# Ensure the repo root is on the import path
_repo_root = os.path.dirname(os.path.abspath("__file__"))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

from config import *
from lseg_api import (
    get_access_token, fetch_reports_info, extract_rows,
    QUERY_BUILDERS, get_retrieval_url,
)
from downloader import BulkDownloader
from ranking import (
    rank_annual_reports, rank_esg_reports,
    get_language_map_df, get_submission_type_df, get_feed_priority_df,
)

from pyspark.sql import functions as F, Window
from datetime import datetime, date

## LSEG Filings Pipeline

Two modes of operation:

1. **Simple mode** — You have a file/list of document IDs → download PDFs directly
2. **Full mode** — Start from a universe CSV → map to LSEG org IDs → fetch metadata → rank → download

Set the `mode` widget below to choose.

In [0]:
dbutils.widgets.dropdown("mode", "simple", ["simple", "full"], "Pipeline mode")
dbutils.widgets.text("doc_ids_path", "", "Doc IDs file path (simple mode)")
dbutils.widgets.text("universe_csv_path", "", "Universe CSV path (full mode)")
dbutils.widgets.text("output_subfolder", REPORT_SUBFOLDER, "Output blob subfolder")

_mode = dbutils.widgets.get("mode")
_doc_ids_path = dbutils.widgets.get("doc_ids_path")
_universe_csv_path = dbutils.widgets.get("universe_csv_path")
_output_subfolder = dbutils.widgets.get("output_subfolder")

print(f"Mode: {_mode}")

In [0]:
if _mode == "simple":
    # Load doc IDs from CSV, Spark table, or provide them inline
    if _doc_ids_path:
        doc_ids_df = spark.read.csv(_doc_ids_path, inferSchema=True, header=True)
        # Expect a column called 'document_id' (or first column)
        id_col = "document_id" if "document_id" in doc_ids_df.columns else doc_ids_df.columns[0]
        doc_ids = [row[0] for row in doc_ids_df.select(id_col).collect()]
    else:
        # Inline example — replace with your IDs
        doc_ids = [
            # 123456789,
            # 987654321,
        ]

    print(f"Loaded {len(doc_ids)} document IDs")

    if doc_ids:
        dl = BulkDownloader(dbutils)
        failed = dl.download_by_ids(doc_ids, subfolder=_output_subfolder)
        print(f"\nFailed IDs: {failed}")
    else:
        print("No document IDs to download. Set doc_ids_path or add IDs inline.")

## Full Pipeline Mode

The cells below run the complete pipeline:
1. Load universe CSV and map Robeco Party IDs → LSEG organization IDs
2. Fetch document metadata from the LSEG GraphQL API
3. Rank and deduplicate to pick the best report per org
4. Bulk download the selected PDFs

Skip these cells if using simple mode.

In [0]:
if _mode == "full":
    assert _universe_csv_path, "Set the universe_csv_path widget for full mode"

    universe_raw = spark.read.csv(_universe_csv_path, inferSchema=True, header=True)
    universe = (
        universe_raw
        .select(
            F.col("Robeco Party ID").cast("long").alias("robeco_party"),
            F.lit(1).alias("si_universe"),
            "company_name",
        )
        .where(F.col("robeco_party").isNotNull())
    )
    print(f"Universe: {universe.count()} parties")
    universe.display()

In [0]:
if _mode == "full":
    from concurrent.futures import ThreadPoolExecutor, as_completed
    from time import monotonic
    from downloader import RateLimiter, TokenProvider

    _today = date.today()
    _from_3y = date(_today.year - 3, _today.month, 1)

    rate_limiter = RateLimiter(REQUESTS_PER_SECOND)
    token_provider = TokenProvider(dbutils)

    def fetch_org_metadata(org_id, mid_category, from_date, to_date):
        """Fetch filing metadata for one org + category."""
        rate_limiter.wait()
        token = token_provider.get()
        query_builder = QUERY_BUILDERS[mid_category]
        query = query_builder(org_id, from_date, to_date)
        response = fetch_reports_info(token, query)
        return extract_rows(org_id, response)

    # Collect org IDs (assumes universe has been mapped to LSEG org IDs)
    # TODO: Add ISIN-based mapping step if your universe doesn't have organization_id
    org_ids = [row["robeco_party"] for row in universe.select("robeco_party").collect()]

    all_rows = []
    for mid_cat in [1, 26]:  # Annual, ESG
        print(f"\nFetching category {mid_cat} for {len(org_ids)} organizations...")
        started = monotonic()
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {
                executor.submit(fetch_org_metadata, oid, mid_cat, _from_3y, _today): oid
                for oid in org_ids
            }
            for future in as_completed(futures):
                oid = futures[future]
                try:
                    rows = future.result()
                    all_rows.extend(rows)
                except Exception as e:
                    print(f"  Failed org {oid}: {e}")
        print(f"  Done in {monotonic() - started:.1f}s, {len(all_rows)} rows total")

    doc_meta_df = spark.createDataFrame(all_rows)
    print(f"\nTotal metadata rows: {doc_meta_df.count()}")
    doc_meta_df.display()

In [0]:
if _mode == "full":
    ranked_annual = rank_annual_reports(doc_meta_df.where(F.col("midlevelcategoryid") == 1), spark)
    ranked_esg = rank_esg_reports(doc_meta_df.where(F.col("midlevelcategoryid") == 26), spark)

    to_download = (
        ranked_annual.select("organization_id", "document_id", "midlevelcategoryid")
        .unionByName(
            ranked_esg.select("organization_id", "document_id", "midlevelcategoryid")
        )
    )

    print(f"Annual: {ranked_annual.count()}, ESG: {ranked_esg.count()}")
    print(f"Total to download: {to_download.count()}")

In [0]:
if _mode == "full":
    language_map = get_language_map_df(spark)

    reports = (
        to_download
        .join(doc_meta_df, ["organization_id", "document_id"])
        .join(language_map, "languageid", "left")
        .join(universe, F.col("organization_id") == F.col("robeco_party"), "left")
        .rdd.map(lambda row: row.asDict())
        .collect()
    )

    print(f"Downloading {len(reports)} reports...")
    dl = BulkDownloader(dbutils)
    failed = dl.download_reports(reports, subfolder=_output_subfolder)
    print(f"\nFailed: {failed}")